In [1]:
import os
import json
import pandas as pd
import yfinance as yf
print(yf.__version__)

1.2.0


In [ ]:
# 1 - DOWNLOAD DER DATEN VON YFINANCE IN datenRAW

In [ ]:
assets = ["^GSPC", "^GDAXI", "^NDX", "AAPL", "PG", "XOM", "JPM", "SIE.DE", "DTE.DE"]
startDate = "2010-01-01"
endDate = "2023-12-31"
outputFolder = "datenRAW"

def downloadAssetData(assets, startDate, endDate, outputFolder):
    for asset in assets:
        data = yf.download(asset, start=startDate, end=endDate)
        data.to_csv(f"{outputFolder}/{asset}_daily.csv")

# downloadAssetData(assets, startDate, endDate, outputFolder)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [ ]:
# 2 - DATENBEREINIGUNG & SPEICHERUNG IN datenCLEAN

In [ ]:
def dataFrameCleaner(df):
    df.rename(columns = {"Price": "Date"}, inplace=True)
    df = df.iloc[2:].copy()
    df.reset_index(drop=True, inplace=True)
    df['Date'] = pd.to_datetime(df['Date'])
    cols = ["Close", "High", "Low", "Open", "Volume"]
    df[cols] = df[cols].apply(pd.to_numeric)
    return df

def createNewFileName(rawDataFileName):
    if "^" in rawDataFileName:
        rawDataFileName = rawDataFileName.replace("^", "")
    newFileName = rawDataFileName.split("_daily.csv")[0] + "_clean.csv"
    return newFileName

def cleanRawFiles(inputFolder, outputFolder):
    rawFiles = os.listdir(inputFolder)

    for rawDataFileName in rawFiles:
        df = pd.read_csv(f"{inputFolder}/{rawDataFileName}")
        cleanDf = dataFrameCleaner(df)
        cleanDataFileName = createNewFileName(rawDataFileName)
        cleanDf.to_csv(f"{outputFolder}/{cleanDataFileName}", index=False)

# cleanRawFiles("datenRAW", "datenCLEAN")

In [ ]:
# 3 - METADATEN

In [ ]:
# KLASSIFIKATION DER MARKTPHASEN

In [ ]:
def readInCsv(filename, folderName):
    return pd.read_csv(f"{folderName}/{filename}", parse_dates=["Date"], index_col="Date")

def createYearSeperation(folderName):
    ALLAssetsYearSeperated = {}
    cleanData = os.listdir(folderName)
    
    for filename in cleanData:

        assetYearSeperated = {}
        assetName = filename.split("_")[0]

        df = readInCsv(filename, folderName)
        years = df.index.year.unique()

        for year in years:
            yearDF = df[df.index.year == year]
            assetYearSeperated[year] = yearDF

        ALLAssetsYearSeperated[assetName] = assetYearSeperated
        
    return ALLAssetsYearSeperated

dictionaryAssetYearData = createYearSeperation(folderName = "datenCLEAN")

def detectMarketTrends(dictionaryAssetYearData):
    assets = dictionaryAssetYearData.keys()
    marketReturnsTrends = {}
    theta = 0.2
    
    for asset in assets:
        
        jahrInfo = {}
        
        for year, df in dictionaryAssetYearData[asset].items():
            startPrice = df["Close"].iloc[0]
            endPrice = df["Close"].iloc[-1]
            yearlyReturn = abs((endPrice - startPrice) / startPrice)
            
            if yearlyReturn >= theta:
                phase = "trend"
            else:
                phase = "sideways"
            
            jahrInfo[year] = {
                "absReturn": float(yearlyReturn),
                "phase": phase
            }
        
        marketReturnsTrends[asset] = jahrInfo
    return marketReturnsTrends

marketReturnsTrends = detectMarketTrends(dictionaryAssetYearData)

# with open("metadata/marketReturnsTrends.json", "w") as f:
#     json.dump(marketReturnsTrends, f)

In [ ]:
# ZUSÄTZLICHE METADATEN

In [ ]:
assetMeta = {
    "AAPL": {
        "assetType": "Stock",
        "market": "USA",
        "currency": "USD"
    },
    "PG": {
        "assetType": "Stock",
        "market": "USA",
        "currency": "USD"
    },
    "XOM": {
        "assetType": "Stock",
        "market": "USA",
        "currency": "USD"
    },
    "JPM": {
        "assetType": "Stock",
        "market": "USA",
        "currency": "USD"
    },
    "SIE.DE": {
        "assetType": "Stock",
        "market": "Europe",
        "currency": "EUR"
    },
    "DTE.DE": {
        "assetType": "Stock",
        "market": "Europe",
        "currency": "EUR"
    },
    "GSPC": {
        "assetType": "MarketIndex",
        "market": "USA",
        "currency": "USD"
    },
    "NDX": {
        "assetType": "MarketIndex",
        "market": "USA",
        "currency": "USD"
    },
    "GDAXI": {
        "assetType": "MarketIndex",
        "market": "Europe",
        "currency": "EUR"
    }
}
# with open("metadata/assetMeta.json", "w") as f:
#     json.dump(assetMeta, f)